In [1]:
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors
from annoy import AnnoyIndex
import hnswlib
import faiss
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Contoh dataset kecil untuk testing
# -------------------------------
np.random.seed(42)
n_samples = 10000   # jumlah database vector
d = 128             # dimensi
X = np.random.random((n_samples, d)).astype('float32')

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

# -------------------------------
# Exact NN (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=d)
p_hnsw.init_index(max_elements=n_samples, ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, _ = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()
quantizer = faiss.IndexFlatL2(d)
index_faiss = faiss.IndexIVFFlat(quantizer, d, 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
_, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

# -------------------------------
# Tampilkan ringkasan waktu
# -------------------------------
print("\n=== Ringkasan Waktu (detik) ===")
print(f"Exact NN : {time_exact:.3f}")
print(f"Annoy    : {time_annoy:.3f}")
print(f"HNSW     : {time_hnsw:.3f}")
print(f"FAISS    : {time_faiss:.3f}")

Exact NN done in 2.095 s


Annoy done in 0.912 s


HNSW done in 0.606 s
FAISS IVF done in 0.114 s

=== Ringkasan Waktu (detik) ===
Exact NN : 2.095
Annoy    : 0.912
HNSW     : 0.606
FAISS    : 0.114


### Spesifikasi Smartphone:
- **Merk / Tipe**: Xiaomi Redmi Note 13 (23124RA7EO)
- **RAM**: 8 GB
- **Chipset**: Qualcomm Snapdragon 685 (SM6225) Octa-core (4x2.8 GHz Cortex-A73 & 4x1.9 GHz Cortex-A53)
- **OS**: Android

### Perbandingan Hasil Waktu Eksekusi:
| Model | Google Colab / PC (s) | PyDroid3 Android (s) | Keterangan |
| :--- | :---: | :---: | :--- |
| Exact NN | 2.095 | 1.297 | Berhasil dieksekusi secara native via scikit-learn |
| Annoy | 0.912 | N/A | Tidak terinstall (keterbatasan build tool C++ di Android) |
| HNSW | 0.606 | N/A | Tidak terinstall (keterbatasan build tool C++ di Android) |
| FAISS IVF | 0.114 | N/A | Tidak terinstall (wheel binary ARM64 tidak tersedia di PyPI) |

### Analisis Hasil Pengujian:
1. **Keterbatasan Ekosistem Library Machine Learning di Mobile**:
   - Berdasarkan hasil eksekusi langsung pada PyDroid3 di Xiaomi Redmi Note 13, library ANN khusus seperti Annoy, HNSW, dan FAISS tidak dapat dijalankan secara langsung (*out-of-the-box*). Hal ini disebabkan library tersebut dibangun menggunakan dependensi C++ tingkat rendah dan belum menyediakan distribusi *precompiled binary wheel* (.whl) untuk platform Android (bionic libc / ARM64).
2. **Performa Algoritma Brute-Force (Exact NN)**:
   - Algoritma Exact NN dari scikit-learn berhasil dieksekusi dengan waktu **1.297 detik** di smartphone (dibandingkan **2.095 detik** di PC). Hal ini menunjukkan bahwa untuk komputasi matriks umum yang sudah teroptimasi pada wheel NumPy/SciPy mobile, arsitektur octa-core Snapdragon 685 mampu menangani perhitungan jarak Euclidean dengan sangat responsif pada ukuran dataset 10.000 sampel.
3. **Tantangan Deployment ANN pada Perangkat Edge**:
   - Pengujian ini membuktikan bahwa implementasi algoritma ANN pada aplikasi mobile membutuhkan pendekatan *cross-compilation* khusus, integrasi C++ native via NDK, atau model yang telah dikonversi ke runtime khusus mobile (seperti ONNX Runtime Mobile atau TensorFlow Lite), alih-alih mengandalkan instalasi runtime Python standar seperti di desktop.